# 1: Install libraries



In [ ]:
!pip install -q transformers datasets evaluate accelerate sentencepiece sacrebleu rouge_score bert_score nltk pandas scikit-learn openpyxl


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.6 MB/s eta 0:00:00


# 2: Import libraries

In [ ]:
import os
import re
import gc
import math
import torch
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    default_data_collator,
    EarlyStoppingCallback
)

import evaluate
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score

nltk.download("wordnet")
nltk.download("omw-1.4")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Device:", device)


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


GPU available: True
GPU: Tesla T4
Device: cuda


# 3: Upload dataset

In [ ]:
from google.colab import files

uploaded = files.upload()
csv_path = list(uploaded.keys())[0]

print("Uploaded file:", csv_path)

Saving Counterspeech_dataset_Old_cleaned (1).csv to Counterspeech_dataset_Old_cleaned (1).csv
Uploaded file: Counterspeech_dataset_Old_cleaned (1).csv


# 4: Load dataset

In [ ]:
df = pd.read_csv(csv_path)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

df.head()

Dataset shape: (3011, 5)
Columns: ['id', 'offensive_text', 'counterspeech_text', 'counterspeech_type', 'text_type']


,id,offensive_text,counterspeech_text,counterspeech_type,text_type
0,1,"কি মনে করছ, মেয়েরা কিছু বোঝে ?",এই কথা ঠিক না। বাংলাদেশে অনেক মেয়ে দক্ষতার সাথ...,fact_based,hate_speech
1,2,"তুই তো বেশ্যা, রাস্তায় খেপ মাইরা বেরাছ",মানুষ কে সম্মান করতে শিখুন। তবেই নিজে সম্মানিত...,warning_of_consequences,sexual_harassment
2,3,হিন্দুরাই দেশের সম্পদ পাচার করতেছে,পুরো একটা গোষ্ঠীকে দোষ দেওয়া ঠিক না।,empathy,hate_speech
3,4,মাইয়া মানুষ আবার গাড়ি চালাইতে পারে নাকি!,এই ধারণা ভুল। বর্তমানে অনেক মেয়েরা সফলভাবে গাড়...,fact_based,gender_abuse
4,5,এই ছাগলের কথা শুনলে মাথাই গরম হয়ে যায়,ভদ্রভাবে কথা বললে আলোচনা আরও সুন্দর হয়।,moral_appeal,abusive_language


# 5: Keep only generation columns

In [ ]:
required_columns = ["offensive_text", "counterspeech_text"]

for col in required_columns:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

df = df[required_columns].copy()

df.head()

,offensive_text,counterspeech_text
0,"কি মনে করছ, মেয়েরা কিছু বোঝে ?",এই কথা ঠিক না। বাংলাদেশে অনেক মেয়ে দক্ষতার সাথ...
1,"তুই তো বেশ্যা, রাস্তায় খেপ মাইরা বেরাছ",মানুষ কে সম্মান করতে শিখুন। তবেই নিজে সম্মানিত...
2,হিন্দুরাই দেশের সম্পদ পাচার করতেছে,পুরো একটা গোষ্ঠীকে দোষ দেওয়া ঠিক না।
3,মাইয়া মানুষ আবার গাড়ি চালাইতে পারে নাকি!,এই ধারণা ভুল। বর্তমানে অনেক মেয়েরা সফলভাবে গাড়...
4,এই ছাগলের কথা শুনলে মাথাই গরম হয়ে যায়,ভদ্রভাবে কথা বললে আলোচনা আরও সুন্দর হয়।


# 6: Clean text

In [ ]:
def clean_text(text):
    text = str(text)
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    text = text.replace("\u200c", "")
    text = text.replace("\u200d", "")
    return text

df["offensive_text"] = df["offensive_text"].apply(clean_text)
df["counterspeech_text"] = df["counterspeech_text"].apply(clean_text)

df = df.dropna()
df = df.drop_duplicates()
df = df[df["offensive_text"].str.len() > 2]
df = df[df["counterspeech_text"].str.len() > 2]

print("Cleaned dataset shape:", df.shape)
df.head()

Cleaned dataset shape: (3011, 2)


,offensive_text,counterspeech_text
0,"কি মনে করছ, মেয়েরা কিছু বোঝে ?",এই কথা ঠিক না। বাংলাদেশে অনেক মেয়ে দক্ষতার সাথ...
1,"তুই তো বেশ্যা, রাস্তায় খেপ মাইরা বেরাছ",মানুষ কে সম্মান করতে শিখুন। তবেই নিজে সম্মানিত...
2,হিন্দুরাই দেশের সম্পদ পাচার করতেছে,পুরো একটা গোষ্ঠীকে দোষ দেওয়া ঠিক না।
3,মাইয়া মানুষ আবার গাড়ি চালাইতে পারে নাকি!,এই ধারণা ভুল। বর্তমানে অনেক মেয়েরা সফলভাবে গাড়...
4,এই ছাগলের কথা শুনলে মাথাই গরম হয়ে যায়,ভদ্রভাবে কথা বললে আলোচনা আরও সুন্দর হয়।


# 7: Create input and target text

In [ ]:
# Keep the raw offensive text here.
# The causal-LM prompt is constructed later inside build_prompt().
df["input_text"] = df["offensive_text"]
df["target_text"] = df["counterspeech_text"]

df = df[["input_text", "target_text"]]

df.head()


,input_text,target_text
0,"কি মনে করছ, মেয়েরা কিছু বোঝে ?",এই কথা ঠিক না। বাংলাদেশে অনেক মেয়ে দক্ষতার সাথ...
1,"তুই তো বেশ্যা, রাস্তায় খেপ মাইরা বেরাছ",মানুষ কে সম্মান করতে শিখুন। তবেই নিজে সম্মানিত...
2,হিন্দুরাই দেশের সম্পদ পাচার করতেছে,পুরো একটা গোষ্ঠীকে দোষ দেওয়া ঠিক না।
3,মাইয়া মানুষ আবার গাড়ি চালাইতে পারে নাকি!,এই ধারণা ভুল। বর্তমানে অনেক মেয়েরা সফলভাবে গাড়...
4,এই ছাগলের কথা শুনলে মাথাই গরম হয়ে যায়,ভদ্রভাবে কথা বললে আলোচনা আরও সুন্দর হয়।


# 8: Train-validation-test split

In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
    shuffle=True
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    shuffle=True
)

print("Train size:", len(train_df))
print("Validation size:", len(valid_df))
print("Test size:", len(test_df))

Train size: 2408
Validation size: 301
Test size: 302


# 9: Convert to Hugging Face Dataset

In [ ]:
dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
    "validation": Dataset.from_pandas(valid_df.reset_index(drop=True)),
    "test": Dataset.from_pandas(test_df.reset_index(drop=True))
})

dataset

DatasetDict({
    train: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 2408
    })
    validation: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 301
    })
    test: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 302
    })
})

# 10: Load BLOOM-560M


In [ ]:
MODEL_NAME = "bigscience/bloom-560m"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

# Force FP32 model weights for numerical stability.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True
)

model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id
model.config.use_cache = False

# IMPORTANT:
# Gradient checkpointing was removed because the manual forward loss was
# finite, while training-mode updates produced NaN values in this runtime.
model.gradient_checkpointing_disable()

model.to(device)

VOCAB_SIZE = model.get_input_embeddings().weight.shape[0]

print("Model:", MODEL_NAME)
print("Device:", device)
print("Model dtype:", next(model.parameters()).dtype)
print("Tokenizer size:", len(tokenizer))
print("Embedding vocabulary size:", VOCAB_SIZE)
print("EOS token:", tokenizer.eos_token, tokenizer.eos_token_id)
print("PAD token:", tokenizer.pad_token, tokenizer.pad_token_id)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/222 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

Model: bigscience/bloom-560m
Device: cuda
Model dtype: torch.float32
Tokenizer size: 250680
Embedding vocabulary size: 250880
EOS token: </s> 2
PAD token: <pad> 3


# 11: Tokenize for Causal Language Modeling


In [ ]:
MAX_PROMPT_LENGTH = 128
MAX_TARGET_LENGTH = 96
MAX_LENGTH = MAX_PROMPT_LENGTH + MAX_TARGET_LENGTH

def build_prompt(offensive_text):
    return (
        "আপত্তিকর বক্তব্য: "
        + str(offensive_text).strip()
        + "\nভদ্র প্রতিউত্তর:"
    )


def preprocess_function(examples):
    input_ids_batch = []
    attention_mask_batch = []
    labels_batch = []

    for offensive_text, target_text in zip(
        examples["input_text"],
        examples["target_text"]
    ):
        prompt = build_prompt(offensive_text)
        target = " " + str(target_text).strip()

        # Tokenize prompt and target separately so prompt tokens can be
        # masked from the causal-language-model loss.
        prompt_ids = tokenizer(
            prompt,
            max_length=MAX_PROMPT_LENGTH,
            truncation=True,
            padding=False,
            add_special_tokens=False
        )["input_ids"]

        target_ids = tokenizer(
            target,
            max_length=MAX_TARGET_LENGTH - 1,
            truncation=True,
            padding=False,
            add_special_tokens=False
        )["input_ids"]

        # Explicitly end each target with EOS.
        target_ids = target_ids + [tokenizer.eos_token_id]

        input_ids = prompt_ids + target_ids
        labels = ([-100] * len(prompt_ids)) + target_ids
        attention_mask = [1] * len(input_ids)

        # Truncate only as a final safety measure.
        input_ids = input_ids[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]

        # Guarantee that the final valid target token is EOS.
        valid_label_positions = [
            idx for idx, token_id in enumerate(labels)
            if token_id != -100
        ]
        if not valid_label_positions:
            raise ValueError(
                "A sample has no valid target tokens after tokenization."
            )

        last_target_position = valid_label_positions[-1]
        input_ids[last_target_position] = tokenizer.eos_token_id
        labels[last_target_position] = tokenizer.eos_token_id

        pad_length = MAX_LENGTH - len(input_ids)

        if pad_length > 0:
            input_ids += [tokenizer.pad_token_id] * pad_length
            attention_mask += [0] * pad_length
            labels += [-100] * pad_length

        # Detect out-of-range token IDs before they reach CUDA.
        for token_id in input_ids:
            if token_id < 0 or token_id >= VOCAB_SIZE:
                raise ValueError(f"Invalid input token ID: {token_id}")

        for token_id in labels:
            if token_id != -100 and (
                token_id < 0 or token_id >= VOCAB_SIZE
            ):
                raise ValueError(f"Invalid label token ID: {token_id}")

        input_ids_batch.append(input_ids)
        attention_mask_batch.append(attention_mask)
        labels_batch.append(labels)

    return {
        "input_ids": input_ids_batch,
        "attention_mask": attention_mask_batch,
        "labels": labels_batch
    }


tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=["input_text", "target_text"]
)

tokenized_dataset


Map:   0%|          | 0/2408 [00:00<?, ? examples/s]

Map:   0%|          | 0/301 [00:00<?, ? examples/s]

Map:   0%|          | 0/302 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2408
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 301
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 302
    })
})

# 12: Debug Tokenization


In [ ]:
sample = tokenized_dataset["train"][0]

unpadded_input_ids = [
    token_id
    for token_id, mask in zip(
        sample["input_ids"],
        sample["attention_mask"]
    )
    if mask == 1
]

valid_label_ids = [
    token_id for token_id in sample["labels"]
    if token_id != -100
]

decoded_full_text = tokenizer.decode(
    unpadded_input_ids,
    skip_special_tokens=False
)

decoded_target = tokenizer.decode(
    valid_label_ids,
    skip_special_tokens=True
)

print("Decoded prompt + target:")
print(decoded_full_text)

print("\nDecoded target labels only:")
print(decoded_target)

print("\nValid target tokens:", len(valid_label_ids))
print("Last target token ID:", valid_label_ids[-1])
print("Expected EOS token ID:", tokenizer.eos_token_id)

assert len(valid_label_ids) > 1
assert valid_label_ids[-1] == tokenizer.eos_token_id


Decoded prompt + target:
আপত্তিকর বক্তব্য: নারীরা এগিয়ে যাচ্ছে, কিন্তু কোথায় সেটা বললে হুরেরা এসে আমার চৌদ্দগুষ্টি তুলে গালাগালি করবে।
ভদ্র প্রতিউত্তর: ভিন্ন মতের মানুষকে হেয় না করে আলোচনা করুন। অসম্মানজনক ভাষা ছাড়া বিতর্ক অনেক বেশি ফলপ্রসূ হয়.</s>

Decoded target labels only:
 ভিন্ন মতের মানুষকে হেয় না করে আলোচনা করুন। অসম্মানজনক ভাষা ছাড়া বিতর্ক অনেক বেশি ফলপ্রসূ হয়.

Valid target tokens: 25
Last target token ID: 2
Expected EOS token ID: 2


# 13: Data Collator


In [ ]:
# Tokenized samples are already padded to the same fixed length.
# This collator preserves the custom -100 prompt masking.
data_collator = default_data_collator


# 14: Batch Loss and Token-Range Test


In [ ]:
from torch.utils.data import DataLoader

# Check every split for empty-label samples before touching the GPU.
for split_name in ["train", "validation", "test"]:
    valid_counts = [
        sum(token_id != -100 for token_id in labels)
        for labels in tokenized_dataset[split_name]["labels"]
    ]

    print(
        split_name,
        "samples:", len(valid_counts),
        "| minimum valid labels:", min(valid_counts),
        "| maximum valid labels:", max(valid_counts)
    )

    if min(valid_counts) < 2:
        raise ValueError(
            f"{split_name} contains a sample with fewer than two "
            "trainable target tokens."
        )

small_train_dataset = tokenized_dataset["train"].select(
    range(min(4, len(tokenized_dataset["train"])))
)

test_loader = DataLoader(
    small_train_dataset,
    batch_size=1,
    collate_fn=data_collator,
    shuffle=False
)

batch = next(iter(test_loader))
valid_labels = batch["labels"][batch["labels"] != -100]

print("\nInput shape:", tuple(batch["input_ids"].shape))
print("Label shape:", tuple(batch["labels"].shape))
print("Valid label tokens:", valid_labels.numel())
print(
    "Input token range:",
    int(batch["input_ids"].min()),
    "to",
    int(batch["input_ids"].max())
)
print(
    "Label token range:",
    int(valid_labels.min()),
    "to",
    int(valid_labels.max())
)
print("Model vocabulary size:", VOCAB_SIZE)

assert valid_labels.numel() > 1
assert int(batch["input_ids"].max()) < VOCAB_SIZE
assert int(valid_labels.max()) < VOCAB_SIZE

batch = {key: value.to(device) for key, value in batch.items()}

# Test in training mode because Trainer uses model.train().
model.train()
model.zero_grad(set_to_none=True)

outputs = model(**batch)
test_loss = outputs.loss

print("Training-mode batch loss:", float(test_loss.item()))

if not torch.isfinite(test_loss) or float(test_loss.item()) <= 0:
    raise ValueError(
        "The training-mode forward loss is invalid. Do not start training."
    )

# Check backward gradients without performing an optimizer update.
test_loss.backward()

nonfinite_gradients = []
gradient_norms = []

for name, parameter in model.named_parameters():
    if parameter.grad is None:
        continue

    if not torch.isfinite(parameter.grad).all():
        nonfinite_gradients.append(name)
    else:
        gradient_norms.append(parameter.grad.detach().float().norm().item())

print("Parameters with non-finite gradients:", len(nonfinite_gradients))
print(
    "Largest finite parameter-gradient norm:",
    max(gradient_norms) if gradient_norms else 0.0
)

model.zero_grad(set_to_none=True)

if nonfinite_gradients:
    print("First problematic parameters:", nonfinite_gradients[:10])
    raise FloatingPointError(
        "The backward pass produced NaN/Inf gradients."
    )

model.eval()
print("Forward and backward diagnostic passed.")


train samples: 2408 | minimum valid labels: 7 | maximum valid labels: 42
validation samples: 301 | minimum valid labels: 9 | maximum valid labels: 36
test samples: 302 | minimum valid labels: 10 | maximum valid labels: 37

Input shape: (1, 224)
Label shape: (1, 224)
Valid label tokens: 25
Input token range: 2 to 184016
Label token range: 2 to 145599
Model vocabulary size: 250880
Training-mode batch loss: 4.02775764465332
Parameters with non-finite gradients: 0
Largest finite parameter-gradient norm: 229.6680908203125
Forward and backward diagnostic passed.


# 15: Training Arguments


In [ ]:
OUTPUT_DIR = "/content/bloom_560m_counterspeech_stable"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    eval_strategy="epoch",
    save_strategy="epoch",

    # Reduced from 2e-5 because the pretrained weights became non-finite
    # after optimizer updates even though the initial forward loss was valid.
    learning_rate=5e-6,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    num_train_epochs=4,

    weight_decay=0.01,
    warmup_ratio=0.05,
    max_grad_norm=0.5,

    # Use explicit, non-fused FP32 AdamW.
    optim="adamw_torch",
    fp16=False,
    bf16=False,
    gradient_checkpointing=False,

    logging_steps=25,

    # Do not hide NaN/Inf losses by displaying them as 0.000000.
    logging_nan_inf_filter=False,

    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to="none",
    seed=SEED,

    dataloader_pin_memory=False,
    remove_unused_columns=False
)


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


# 16: Create Trainer


In [ ]:
class FiniteLossTrainer(Trainer):
    """
    Calls BLOOM directly and immediately stops if a batch loss becomes
    NaN/Inf. It also avoids version-dependent loss rescaling based on
    num_items_in_batch.
    """

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None
    ):
        outputs = model(**inputs)
        loss = outputs.loss

        if loss is None:
            raise RuntimeError("The BLOOM model did not return a loss.")

        if not torch.isfinite(loss):
            valid_tokens = int((inputs["labels"] != -100).sum().item())
            raise FloatingPointError(
                "Non-finite BLOOM loss detected before the optimizer step. "
                f"Valid target tokens in this batch: {valid_tokens}"
            )

        return (loss, outputs) if return_outputs else loss


trainer = FiniteLossTrainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],

    processing_class=tokenizer,
    data_collator=data_collator,

    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=2)
    ]
)

# Validation must be finite before training begins.
initial_evaluation = trainer.evaluate()
print("Initial validation result:", initial_evaluation)

initial_eval_loss = float(initial_evaluation["eval_loss"])

if not math.isfinite(initial_eval_loss) or initial_eval_loss <= 0:
    raise RuntimeError(
        "Initial validation loss is invalid before training."
    )


Training Loss,Validation Loss,Epoch
No log,4.699174,0


Initial validation result: {'eval_loss': 4.699173927307129}


# 17: Train BLOOM-560M


In [ ]:
train_output = trainer.train()

print(train_output)

reported_train_loss = float(train_output.training_loss)

if not math.isfinite(reported_train_loss) or reported_train_loss <= 0:
    raise RuntimeError(
        "BLOOM training failed: final training loss is zero, NaN, "
        "or infinite."
    )


Epoch,Training Loss,Validation Loss
1,17.783073,2.029461
2,13.517963,1.888361
3,10.191075,1.915289
4,6.797068,2.030080


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=1204, training_loss=12.973424075053776, metrics={'train_runtime': 4144.8585, 'train_samples_per_second': 2.324, 'train_steps_per_second': 0.29, 'total_flos': 3913571238936576.0, 'train_loss': 12.973424075053776, 'epoch': 4.0})


# 18: Evaluate Validation Loss


In [ ]:
evaluation_result = trainer.evaluate()
print(evaluation_result)

evaluation_loss = float(evaluation_result["eval_loss"])

if not math.isfinite(evaluation_loss) or evaluation_loss <= 0:
    raise RuntimeError(
        "BLOOM validation loss is zero, NaN, or infinite."
    )


Training Loss,Validation Loss,Epoch
6.797068,1.888361,4


{'eval_loss': 1.8883609771728516}


# 19: Save Fine-Tuned BLOOM Model


In [ ]:
FINAL_MODEL_DIR = "/content/final_bloom_560m_counterspeech_stable"

trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print("Model saved to:", FINAL_MODEL_DIR)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: /content/final_bloom_560m_counterspeech_stable


# 20: Manual Generation Test


In [ ]:
model.config.use_cache = True
model.eval()

def clean_generated_text(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)

    # Stop if the model starts another prompt/example.
    stop_markers = [
        "আপত্তিকর বক্তব্য:",
        "ভদ্র প্রতিউত্তর:",
        "Offensive text:",
        "Counterspeech:"
    ]

    for marker in stop_markers:
        if marker in text:
            text = text.split(marker, 1)[0].strip()

    return text


def generate_counterspeech(offensive_text, seed=SEED):
    prompt = build_prompt(offensive_text)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=MAX_PROMPT_LENGTH,
        truncation=True,
        add_special_tokens=False
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    input_length = inputs["input_ids"].shape[1]

    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],

            max_new_tokens=64,
            min_new_tokens=6,

            do_sample=True,
            temperature=0.80,
            top_k=40,
            top_p=0.92,

            repetition_penalty=1.20,
            no_repeat_ngram_size=3,

            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,

            remove_invalid_values=True,
            renormalize_logits=True
        )

    # Decode only newly generated tokens.
    generated_ids = output_ids[0][input_length:]

    output_text = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    )

    return clean_generated_text(output_text)


preview_predictions = []

for i in range(min(20, len(test_df))):
    offensive_text = test_df["input_text"].iloc[i]
    reference = test_df["target_text"].iloc[i]

    prediction = generate_counterspeech(
        offensive_text,
        seed=SEED + i
    )

    preview_predictions.append(prediction)

    print("Offensive:", offensive_text)
    print("Reference:", reference)
    print("Generated:", prediction)
    print("Generated length:", len(prediction.split()))
    print("-" * 80)

unique_count = len(set(preview_predictions))
unique_rate = unique_count / len(preview_predictions)
empty_count = sum(
    1 for text in preview_predictions
    if not str(text).strip()
)

print("Unique preview outputs:", unique_count, "/", len(preview_predictions))
print("Unique preview rate:", unique_rate)
print("Empty preview outputs:", empty_count)

if unique_rate < 0.30:
    print(
        "WARNING: BLOOM may be producing repetitive or mode-collapsed outputs."
    )


Offensive: কি মনে করছ, মেয়েরা কিছু বোঝে ?
Reference: এই কথা ঠিক না। বাংলাদেশে অনেক মেয়ে দক্ষতার সাথে কাজ করছে।
Generated: কারও প্রতি অপমানজনক ভাষা না ব্যবহার করে নির্দিষ্ট আচরণ নিয়ে কথা বলা উচিত।
Generated length: 13
--------------------------------------------------------------------------------
Offensive: আরে আরে ইবলিশের ছোট ভাই বলিস কি তুই! প্রতারক মূল জিনিস এড়িয়ে যাস!
Reference: প্রতারক বললে সমস্যার সমাধান হয় না; নির্দিষ্ট আচরণ উল্লেখ করলে কথা শক্ত হয়।
Generated: এভাবে কাউকে অপমান করলে তার বিশ্বাসযোগ্যতা কমে যায়, তাই শান্তভাবে কথা বলা ভালো।
Generated length: 13
--------------------------------------------------------------------------------
Offensive: নষ্ট সে নষ্টদের তো সাথে থাকবে, এটাই স্বভাবিক।
Reference: মানুষকে নষ্ট বলে দাগিয়ে দিলে বোঝাপড়া কমে যায়, ভদ্র ভাষাই ভালো।
Generated: কাউকে নষ্ট বলা ঘৃণা ও অশ্রদ্ধার কারণ।
Generated length: 7
--------------------------------------------------------------------------------
Offensive: ভারতীয় মানুষ গুলোই শুকরের মত আচরনের জাতি
Reference: এ

# 21: Generate Full Test Predictions


In [ ]:
test_inputs = test_df["input_text"].tolist()
references = test_df["target_text"].tolist()

predictions = []

for i, offensive_text in enumerate(test_inputs):
    prediction = generate_counterspeech(
        offensive_text,
        seed=SEED + i
    )

    # Do not insert a rule-based fallback. Empty or poor outputs must remain
    # visible so that the evaluation honestly reflects BLOOM performance.
    predictions.append(prediction)

    if i % 50 == 0:
        print(f"Generated {i}/{len(test_inputs)}")

results_df = pd.DataFrame({
    "input_text": test_inputs,
    "reference_counterspeech": references,
    "generated_counterspeech": predictions
})

results_df["generated_length"] = results_df[
    "generated_counterspeech"
].apply(lambda text: len(str(text).split()))

unique_output_rate = (
    results_df["generated_counterspeech"].nunique()
    / len(results_df)
)

empty_output_rate_preview = (
    results_df["generated_counterspeech"]
    .astype(str)
    .str.strip()
    .eq("")
    .mean()
)

print("Unique output rate:", unique_output_rate)
print("Empty output rate:", empty_output_rate_preview)

if unique_output_rate < 0.30:
    print(
        "WARNING: The full BLOOM output set has low diversity."
    )

results_df.head(10)


Generated 0/302
Generated 50/302
Generated 100/302
Generated 150/302
Generated 200/302
Generated 250/302
Generated 300/302
Unique output rate: 0.9768211920529801
Empty output rate: 0.0


,input_text,reference_counterspeech,generated_counterspeech,generated_length
0,"কি মনে করছ, মেয়েরা কিছু বোঝে ?",এই কথা ঠিক না। বাংলাদেশে অনেক মেয়ে দক্ষতার সাথ...,কারও প্রতি অপমানজনক ভাষা না ব্যবহার করে নির্দি...,13
1,আরে আরে ইবলিশের ছোট ভাই বলিস কি তুই! প্রতারক ম...,প্রতারক বললে সমস্যার সমাধান হয় না; নির্দিষ্ট আ...,এভাবে কাউকে অপমান করলে তার বিশ্বাসযোগ্যতা কমে ...,13
2,"নষ্ট সে নষ্টদের তো সাথে থাকবে, এটাই স্বভাবিক।","মানুষকে নষ্ট বলে দাগিয়ে দিলে বোঝাপড়া কমে যায়, ...",কাউকে নষ্ট বলা ঘৃণা ও অশ্রদ্ধার কারণ।,7
3,ভারতীয় মানুষ গুলোই শুকরের মত আচরনের জাতি,একটি জাতিকে শুয়োরের সঙ্গে তুলনা করা ঘৃণা ছড়ায়।,মানুষকে গালি না দিয়ে আচরণ নিয়ে কথা বলাই ভালো।,9
4,নাস্তিক ব্লগারদের সাথে তর্ক বিতর্ক করে লাভ নাই...,বিশ্বাসভিন্ন মানুষদের নিয়ে এভাবে বিদ্রূপ করা ...,"প্রমাণ ছাড়া এমন কথা বললে, ভুল ধারণা জন্ম নেয়।",9
5,মোবাইলগুলো কেড়ে নিয়ে ওই বেডিদের পিছনে লাঠি ভরে...,এভাবে আঘাতের কথা বলা আরও ক্ষতি বাড়াবে।,শান্তির কথা বললেও শারীরিক ক্ষতির কথা বলা ঠিক নয়।,9
6,ওইটা আমার বাপ আর তোর আম্মুর বোন রেন্ডির ছেলে,কারও পরিবার নিয়ে এমন কটাক্ষ খুবই অশালীন এবং কষ...,কারও মা-বোন নিয়ে এমন মন্তব্য খুবই অপমানজনক।,7
7,এই মাদকাসক্ত পাটোয়ারীর পিছনে যেসব কথিত তাওহীদ...,কারও মতকে সমর্থন করলেই তাকে বুদ্ধি প্রতিবন্ধী ...,"একটি মতের সবাইকে একসাথে দোষ দিলে, সত্যটা আড়াল ...",10
8,এসব জানোয়ারদের মরন হয় না?,"কারও মৃত্যু কামনা করা মানসিক ক্ষতি ডেকে আনে, ত...","কারও মৃত্যু কামনা করা খুবই আঘাতদায়ক, তাই এ ধরন...",14
9,তুই একটা পাঠান সৈনিক গো বীর্যে জম্ম সেটাই প্রম...,জাতিগত অপমান বাদ দিলে তর্কটা অনেক বেশি পরিষ্কা...,কারও বিরুদ্ধে অভিযোগ থাকলে প্রমাণসহ কথা বলা উচিত।,8


# 22: Check Generation Quality


In [ ]:
print(results_df["generated_length"].describe())

print(
    "Unique generated outputs:",
    results_df["generated_counterspeech"].nunique(),
    "/",
    len(results_df)
)

print(
    "Unique output rate:",
    results_df["generated_counterspeech"].nunique()
    / len(results_df)
)

print(
    "Empty outputs:",
    results_df["generated_counterspeech"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

results_df[
    [
        "reference_counterspeech",
        "generated_counterspeech",
        "generated_length"
    ]
].head(20)


count    302.000000
mean      10.218543
std        2.574844
min        4.000000
25%        8.000000
50%       10.000000
75%       12.000000
max       19.000000
Name: generated_length, dtype: float64
Unique generated outputs: 295 / 302
Unique output rate: 0.9768211920529801
Empty outputs: 0


,reference_counterspeech,generated_counterspeech,generated_length
0,এই কথা ঠিক না। বাংলাদেশে অনেক মেয়ে দক্ষতার সাথ...,কারও প্রতি অপমানজনক ভাষা না ব্যবহার করে নির্দি...,13
1,প্রতারক বললে সমস্যার সমাধান হয় না; নির্দিষ্ট আ...,এভাবে কাউকে অপমান করলে তার বিশ্বাসযোগ্যতা কমে ...,13
2,"মানুষকে নষ্ট বলে দাগিয়ে দিলে বোঝাপড়া কমে যায়, ...",কাউকে নষ্ট বলা ঘৃণা ও অশ্রদ্ধার কারণ।,7
3,একটি জাতিকে শুয়োরের সঙ্গে তুলনা করা ঘৃণা ছড়ায়।,মানুষকে গালি না দিয়ে আচরণ নিয়ে কথা বলাই ভালো।,9
4,বিশ্বাসভিন্ন মানুষদের নিয়ে এভাবে বিদ্রূপ করা ...,"প্রমাণ ছাড়া এমন কথা বললে, ভুল ধারণা জন্ম নেয়।",9
5,এভাবে আঘাতের কথা বলা আরও ক্ষতি বাড়াবে।,শান্তির কথা বললেও শারীরিক ক্ষতির কথা বলা ঠিক নয়।,9
6,কারও পরিবার নিয়ে এমন কটাক্ষ খুবই অশালীন এবং কষ...,কারও মা-বোন নিয়ে এমন মন্তব্য খুবই অপমানজনক।,7
7,কারও মতকে সমর্থন করলেই তাকে বুদ্ধি প্রতিবন্ধী ...,"একটি মতের সবাইকে একসাথে দোষ দিলে, সত্যটা আড়াল ...",10
8,"কারও মৃত্যু কামনা করা মানসিক ক্ষতি ডেকে আনে, ত...","কারও মৃত্যু কামনা করা খুবই আঘাতদায়ক, তাই এ ধরন...",14
9,জাতিগত অপমান বাদ দিলে তর্কটা অনেক বেশি পরিষ্কা...,কারও বিরুদ্ধে অভিযোগ থাকলে প্রমাণসহ কথা বলা উচিত।,8


# 23: Save Predictions


In [ ]:
prediction_csv = "/content/bloom_560m_test_predictions.csv"

results_df.to_csv(
    prediction_csv,
    index=False,
    encoding="utf-8-sig"
)

print("Saved predictions:", prediction_csv)
files.download(prediction_csv)


Saved predictions: /content/bloom_560m_test_predictions.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 24: Metric helper functions

In [ ]:
smooth = SmoothingFunction().method1

def tokenize_bn(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text.split()


def calculate_bleu_scores(predictions, references):
    bleu_1_scores = []
    bleu_2_scores = []
    bleu_3_scores = []
    bleu_4_scores = []

    for pred, ref in zip(predictions, references):
        pred_tokens = tokenize_bn(pred)
        ref_tokens = tokenize_bn(ref)

        if len(pred_tokens) == 0 or len(ref_tokens) == 0:
            bleu_1_scores.append(0)
            bleu_2_scores.append(0)
            bleu_3_scores.append(0)
            bleu_4_scores.append(0)
            continue

        bleu_1_scores.append(sentence_bleu(
            [ref_tokens],
            pred_tokens,
            weights=(1, 0, 0, 0),
            smoothing_function=smooth
        ))

        bleu_2_scores.append(sentence_bleu(
            [ref_tokens],
            pred_tokens,
            weights=(0.5, 0.5, 0, 0),
            smoothing_function=smooth
        ))

        bleu_3_scores.append(sentence_bleu(
            [ref_tokens],
            pred_tokens,
            weights=(1/3, 1/3, 1/3, 0),
            smoothing_function=smooth
        ))

        bleu_4_scores.append(sentence_bleu(
            [ref_tokens],
            pred_tokens,
            weights=(0.25, 0.25, 0.25, 0.25),
            smoothing_function=smooth
        ))

    return {
        "BLEU-1": float(np.mean(bleu_1_scores)),
        "BLEU-2": float(np.mean(bleu_2_scores)),
        "BLEU-3": float(np.mean(bleu_3_scores)),
        "BLEU-4": float(np.mean(bleu_4_scores))
    }


def calculate_meteor(predictions, references):
    scores = []

    for pred, ref in zip(predictions, references):
        pred_tokens = tokenize_bn(pred)
        ref_tokens = tokenize_bn(ref)

        if len(pred_tokens) == 0 or len(ref_tokens) == 0:
            scores.append(0)
            continue

        try:
            score = meteor_score([ref_tokens], pred_tokens)
        except:
            score = 0

        scores.append(score)

    return float(np.mean(scores))


def get_ngrams(tokens, n):
    return list(zip(*[tokens[i:] for i in range(n)]))


def calculate_diversity(predictions):
    all_unigrams = []
    all_bigrams = []

    for text in predictions:
        tokens = tokenize_bn(text)
        all_unigrams.extend(tokens)
        all_bigrams.extend(get_ngrams(tokens, 2))

    distinct_1 = len(set(all_unigrams)) / len(all_unigrams) if len(all_unigrams) > 0 else 0
    distinct_2 = len(set(all_bigrams)) / len(all_bigrams) if len(all_bigrams) > 0 else 0

    diversity = (distinct_1 + distinct_2) / 2

    return {
        "Distinct-1": distinct_1,
        "Distinct-2": distinct_2,
        "Diversity": diversity
    }


# ============================================================
# Improved Novelty Calculation
# Old method: exact copy only
# New method: similarity-based novelty
# Novelty = 1 - Average Maximum Similarity with Training Targets
# ============================================================

def normalize_bn_text(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text


def token_jaccard_similarity(a, b):
    a_tokens = set(tokenize_bn(normalize_bn_text(a)))
    b_tokens = set(tokenize_bn(normalize_bn_text(b)))

    if len(a_tokens) == 0 or len(b_tokens) == 0:
        return 0.0

    intersection = len(a_tokens.intersection(b_tokens))
    union = len(a_tokens.union(b_tokens))

    return intersection / union if union > 0 else 0.0


def calculate_novelty(predictions, train_targets, similarity_threshold=0.70):
    train_targets = [normalize_bn_text(x) for x in train_targets]
    predictions = [normalize_bn_text(x) for x in predictions]

    max_similarities = []
    near_copy_count = 0

    for pred in predictions:
        if len(pred.strip()) == 0:
            max_similarities.append(0.0)
            continue

        similarities = [
            token_jaccard_similarity(pred, train_text)
            for train_text in train_targets
        ]

        max_similarity = max(similarities)
        max_similarities.append(max_similarity)

        if max_similarity >= similarity_threshold:
            near_copy_count += 1

    average_max_similarity = float(np.mean(max_similarities)) if len(max_similarities) > 0 else 0.0
    near_copy_rate = near_copy_count / len(predictions) if len(predictions) > 0 else 0.0

    novelty = 1 - average_max_similarity

    return {
        "Novelty": novelty,
        "Average Max Train Similarity": average_max_similarity,
        "Near Copy Rate": near_copy_rate
    }


bangla_abuse_words = [
    "চুদ",
    "মাগী",
    "হারামি",
    "শালা",
    "কুত্তা",
    "বাল",
    "খানকি",
    "বেশ্যা",
    "মাদারচোদ",
    "বোকাচোদা"
]


def calculate_abuse_word_rate(predictions, abuse_words):
    count = 0

    for text in predictions:
        text = str(text)
        found = any(word in text for word in abuse_words)

        if found:
            count += 1

    abuse_rate = count / len(predictions) if len(predictions) > 0 else 0

    return {
        "Abuse-word rate": abuse_rate,
        "Abusive outputs": count
    }


def calculate_length_stats(predictions):
    lengths = [len(tokenize_bn(text)) for text in predictions]

    empty_count = sum(1 for text in predictions if len(str(text).strip()) == 0)
    empty_rate = empty_count / len(predictions) if len(predictions) > 0 else 0

    return {
        "Average Generated Length": float(np.mean(lengths)),
        "Empty Output Rate": empty_rate
    }

# 25: Bengali ROUGE

In [ ]:
def simple_tokenize_bn(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text.split()


def get_ngrams_list(tokens, n):
    if len(tokens) < n:
        return []
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]


def rouge_n_score(pred, ref, n=1):
    pred_tokens = simple_tokenize_bn(pred)
    ref_tokens = simple_tokenize_bn(ref)

    pred_ngrams = get_ngrams_list(pred_tokens, n)
    ref_ngrams = get_ngrams_list(ref_tokens, n)

    if len(pred_ngrams) == 0 or len(ref_ngrams) == 0:
        return 0.0

    pred_counts = {}
    ref_counts = {}

    for ng in pred_ngrams:
        pred_counts[ng] = pred_counts.get(ng, 0) + 1

    for ng in ref_ngrams:
        ref_counts[ng] = ref_counts.get(ng, 0) + 1

    overlap = 0

    for ng in ref_counts:
        overlap += min(ref_counts[ng], pred_counts.get(ng, 0))

    recall = overlap / len(ref_ngrams)
    precision = overlap / len(pred_ngrams)

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)


def lcs_length(x, y):
    m, n = len(x), len(y)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m):
        for j in range(n):
            if x[i] == y[j]:
                dp[i + 1][j + 1] = dp[i][j] + 1
            else:
                dp[i + 1][j + 1] = max(dp[i][j + 1], dp[i + 1][j])

    return dp[m][n]


def rouge_l_score(pred, ref):
    pred_tokens = simple_tokenize_bn(pred)
    ref_tokens = simple_tokenize_bn(ref)

    if len(pred_tokens) == 0 or len(ref_tokens) == 0:
        return 0.0

    lcs = lcs_length(pred_tokens, ref_tokens)

    recall = lcs / len(ref_tokens)
    precision = lcs / len(pred_tokens)

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)


def calculate_custom_rouge(predictions, references):
    rouge1_scores = []
    rouge2_scores = []
    rougeL_scores = []

    for pred, ref in zip(predictions, references):
        rouge1_scores.append(rouge_n_score(pred, ref, n=1))
        rouge2_scores.append(rouge_n_score(pred, ref, n=2))
        rougeL_scores.append(rouge_l_score(pred, ref))

    return {
        "rouge1": float(np.mean(rouge1_scores)),
        "rouge2": float(np.mean(rouge2_scores)),
        "rougeL": float(np.mean(rougeL_scores))
    }


rouge_scores = calculate_custom_rouge(predictions, references)

rouge_scores

{'rouge1': 0.12138857163930895,
 'rouge2': 0.027950701054821833,
 'rougeL': 0.11372363802038779}

# 26: BERTScore

In [ ]:
!pip install -q bert_score

from bert_score import score as bert_score

clean_predictions = [
    str(x).strip() if str(x).strip() else "ফাঁকা"
    for x in predictions
]

clean_references = [
    str(x).strip() if str(x).strip() else "ফাঁকা"
    for x in references
]

try:
    P, R, F1 = bert_score(
        cands=clean_predictions,
        refs=clean_references,
        model_type="xlm-roberta-base",
        lang="bn",
        rescale_with_baseline=False,
        verbose=True,
        batch_size=16,
        use_fast_tokenizer=True
    )
except Exception as error:
    print("XLM-R BERTScore failed:", error)
    print("Using bert-base-multilingual-cased as fallback.")

    P, R, F1 = bert_score(
        cands=clean_predictions,
        refs=clean_references,
        model_type="bert-base-multilingual-cased",
        lang="bn",
        rescale_with_baseline=False,
        verbose=True,
        batch_size=16,
        use_fast_tokenizer=True
    )

bertscore_precision = float(P.mean().item())
bertscore_recall = float(R.mean().item())
bertscore_f1 = float(F1.mean().item())

print("BERTScore Precision:", bertscore_precision)
print("BERTScore Recall:", bertscore_recall)
print("BERTScore F1:", bertscore_f1)


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/38 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/19 [00:00<?, ?it/s]

done in 1.27 seconds, 238.48 sentences/sec
BERTScore Precision: 0.871476948261261
BERTScore Recall: 0.8649353384971619
BERTScore F1: 0.8681230545043945


# 27: Calculate Final BLOOM Metrics


In [ ]:
bleu_scores = calculate_bleu_scores(predictions, references)

meteor = calculate_meteor(predictions, references)

diversity_scores = calculate_diversity(predictions)

novelty_scores = calculate_novelty(
    predictions,
    train_df["target_text"].tolist(),
    similarity_threshold=0.70
)

abuse_scores = calculate_abuse_word_rate(
    predictions,
    bangla_abuse_words
)

length_scores = calculate_length_stats(predictions)

final_metrics = {
    "Model": "BLOOM-560M",

    "BLEU-1": bleu_scores["BLEU-1"],
    "BLEU-2": bleu_scores["BLEU-2"],
    "BLEU-3": bleu_scores["BLEU-3"],
    "BLEU-4": bleu_scores["BLEU-4"],

    "ROUGE-1": rouge_scores["rouge1"],
    "ROUGE-2": rouge_scores["rouge2"],
    "ROUGE-L": rouge_scores["rougeL"],

    "BERTScore Precision": bertscore_precision,
    "BERTScore Recall": bertscore_recall,
    "BERTScore F1": bertscore_f1,

    "Diversity": diversity_scores["Diversity"],
    "Distinct-1": diversity_scores["Distinct-1"],
    "Distinct-2": diversity_scores["Distinct-2"],

    "Novelty": novelty_scores["Novelty"],
    "Average Max Train Similarity": novelty_scores["Average Max Train Similarity"],
    "Near Copy Rate": novelty_scores["Near Copy Rate"],

    "Abuse-word rate": abuse_scores["Abuse-word rate"],
    "Abusive outputs": abuse_scores["Abusive outputs"],

    "METEOR": meteor,

    "Average Generated Length": length_scores["Average Generated Length"],
    "Empty Output Rate": length_scores["Empty Output Rate"]
}

metrics_df = pd.DataFrame([final_metrics])

metrics_df


,Model,BLEU-1,BLEU-2,BLEU-3,BLEU-4,ROUGE-1,ROUGE-2,ROUGE-L,BERTScore Precision,BERTScore Recall,...,Distinct-1,Distinct-2,Novelty,Average Max Train Similarity,Near Copy Rate,Abuse-word rate,Abusive outputs,METEOR,Average Generated Length,Empty Output Rate
0,BLOOM-560M,0.105805,0.05028,0.030466,0.02181,0.121389,0.027951,0.113724,0.871477,0.864935,...,0.172391,0.479167,0.520136,0.479864,0.076159,0.006623,2,0.079885,10.218543,0.0


# 28: Save BLOOM Metrics


In [ ]:
metrics_csv = "/content/bloom_560m_final_metrics.csv"

metrics_df.to_csv(
    metrics_csv,
    index=False,
    encoding="utf-8-sig"
)

print("Saved metrics:", metrics_csv)
files.download(metrics_csv)


Saved metrics: /content/bloom_560m_final_metrics.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 29: Zip and Download BLOOM Model


In [ ]:
!zip -r /content/final_bloom_560m_counterspeech_stable.zip /content/final_bloom_560m_counterspeech_stable

files.download("/content/final_bloom_560m_counterspeech_stable.zip")


  adding: content/final_bloom_560m_counterspeech_stable/ (stored 0%)
  adding: content/final_bloom_560m_counterspeech_stable/training_args.bin (deflated 53%)
  adding: content/final_bloom_560m_counterspeech_stable/generation_config.json (deflated 29%)
  adding: content/final_bloom_560m_counterspeech_stable/config.json (deflated 50%)
  adding: content/final_bloom_560m_counterspeech_stable/model.safetensors (deflated 7%)
  adding: content/final_bloom_560m_counterspeech_stable/tokenizer.json (deflated 79%)
  adding: content/final_bloom_560m_counterspeech_stable/tokenizer_config.json (deflated 44%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>